# Preparación del Dataset: Adult Census Income

## 1. Descripción del Dataset
Datos del censo para predecir si una persona gana más o menos de 50K al año basado en educación, edad y trabajo.

Aquí procesaremos y prepararemos los datos matemáticamente desde cero usando fórmulas puras, tal como se hace en el curso. Explicaremos paso a paso el código.


## 2. Importación de Librerías
Vamos a cargar las herramientas necesarias, usando lo más básico posible:

- `pandas` y `numpy`: para leer el archivo y manipular la tabla matemáticamente.
- `matplotlib.pyplot` y `seaborn`: para hacer gráficos simples en 2D (líneas, barras), nada en 3D para no complicarlo.

*Nota:* No usaremos funciones complejas o cajas negras para normalizar ni para dividir los datos, todo lo haremos "a mano" con simples matemáticas.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NO usamos sklearn para transformar ni dividir datos.


## 3. Cargar y Explorar el Archivo (Dataset)
Aquí leemos el archivo base y mostramos cómo se ve. Revisamos la cantidad de filas y columnas que tenemos para entender su tamaño.


In [4]:
ruta = 'REPOSITORIOS_PARCIAL/5_Adult (Census Income) – UCI/adult/adult.data'

tabla_datos = pd.read_csv(ruta, header=None, na_values=' ?')

# Mostramos tamaño (filas, columnas)
print('Tamaño del dataset:', tabla_datos.shape)

# Vemos las primeras 5 filas
tabla_datos.head()

# Imprimir información detallada del dataset
filas, columnas = tabla_datos.shape
print(f"\n--- INFORMACIÓN DEL DATASET ---")
print(f"Total de Filas (Ejemplos): {filas}")
print(f"Total de Columnas (Características): {columnas}\n")


Tamaño del dataset: (32561, 15)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 4. Análisis de Datos Vacíos o Faltantes
En la vida real, los datos no siempre vienen perfectos, hay casillas vacías (Nulos o NaN). 

En este caso, hemos decidido **eliminar la fila completa** si detectamos que tiene algún dato faltante. Esto asegura que la red neuronal solo aprenda de casos reales y 100% verídicos (no inventamos promedios ni asumimos valores vacíos), a costa de reducir la cantidad de ejemplos disponibles.


In [5]:
datos_nulos = tabla_datos.isnull().sum()
print('Columnas con casillas vacías antes de limpieza:')
print(datos_nulos[datos_nulos > 0])

# Eliminamos absolutamente cualquier fila que contenga un dato nulo (NaN)
filas_antes = len(tabla_datos)
tabla_datos = tabla_datos.dropna()
filas_despues = len(tabla_datos)

print(f"\nSe eliminaron {filas_antes - filas_despues} filas que contenían datos nulos.")
print('Nulos restantes sumados:', tabla_datos.isnull().sum().sum())


Columnas con casillas vacías:
1     1836
6     1843
13     583
dtype: int64

Nulos restantes sumados: 0


C:\Users\alfao\AppData\Local\Temp\ipykernel_10476\885227427.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = tabla_datos.select_dtypes(include=['object']).columns


## 5. Conversión de Textos a Números
Como la computadora o nuestra Red Neuronal no entiende palabras , necesitamos números. Usaremos una herramienta integrada muy sencilla llamada `get_dummies`.

Lo que hace es tomar las columnas de texto y crear columnas visuales llenas de ceros (`0`) y unos (`1`). Es directo y no necesitas complicarlo con diccionarios o For loops.


In [6]:
# Convertimos todo el texto a datos de 0 y 1 de forma directa y sencilla.
tabla_datos = pd.get_dummies(tabla_datos, dtype=int)

# Mostramos cómo quedó la tabla ahora, lista solo con formato numérico
tabla_datos.head()


,0,2,4,10,11,12,1_ Federal-gov,1_ Local-gov,1_ Never-worked,1_ Private,...,13_ South,13_ Taiwan,13_ Thailand,13_ Trinadad&Tobago,13_ United-States,13_ Vietnam,13_ Yugoslavia,13_Desconocido,14_ <=50K,14_ >50K
0,39,77516,13,2174,0,40,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
1,50,83311,13,0,0,13,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2,38,215646,9,0,0,40,0,0,0,1,...,0,0,0,0,1,0,0,0,1,0
3,53,234721,7,0,0,40,0,0,0,1,...,0,0,0,0,1,0,0,0,1,0
4,28,338409,13,0,0,40,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0


## 6. Normalización de características (Feature Normalize)
Al visualizar los datos se puede observar que las características tienen diferentes magnitudes (por ejemplo, precios altísimos y números pequeños).

**La fórmula para calcular esto con escalado Z-Score es:**
$$ X_{norm} = \frac{X - \mu}{\sigma} $$

En código lo aplicamos usando las operaciones directas `mean()` (media) y `std()` (desviación estándar).


In [7]:
# 1. Calculamos la media (mu) y desviación estándar (sigma) para cada una de las columnas
mu = tabla_datos.mean()
sigma = tabla_datos.std()

# Para evitar errores por dividir entre cero (si la desviación es 0 en alguna columna), forzamos que sea mínimo 1
sigma_seguro = np.where(sigma == 0, 1, sigma)

# 2. Aplicamos la fórmula matemática exacta: (X - mu) / sigma
datos_normalizados = (tabla_datos - mu) / sigma_seguro

# Mostrar la tabla final convertida
datos_normalizados.head()


,0,2,4,10,11,12,1_ Federal-gov,1_ Local-gov,1_ Never-worked,1_ Private,...,13_ South,13_ Taiwan,13_ Thailand,13_ Trinadad&Tobago,13_ United-States,13_ Vietnam,13_ Yugoslavia,13_Desconocido,14_ <=50K,14_ >50K
0,0.030670,-1.063594,1.134721,0.148451,-0.216656,-0.035429,-0.174292,-0.262093,-0.014664,-1.516769,...,-0.049628,-0.039607,-0.023518,-0.024163,0.340949,-0.045408,-0.022172,-0.135021,0.56319,-0.56319
1,0.837096,-1.008692,1.134721,-0.145918,-0.216656,-2.222119,-0.174292,-0.262093,-0.014664,-1.516769,...,-0.049628,-0.039607,-0.023518,-0.024163,0.340949,-0.045408,-0.022172,-0.135021,0.56319,-0.56319
2,-0.042641,0.245075,-0.420053,-0.145918,-0.216656,-0.035429,-0.174292,-0.262093,-0.014664,0.659276,...,-0.049628,-0.039607,-0.023518,-0.024163,0.340949,-0.045408,-0.022172,-0.135021,0.56319,-0.56319
3,1.057031,0.425795,-1.197440,-0.145918,-0.216656,-0.035429,-0.174292,-0.262093,-0.014664,0.659276,...,-0.049628,-0.039607,-0.023518,-0.024163,0.340949,-0.045408,-0.022172,-0.135021,0.56319,-0.56319
4,-0.775756,1.408154,1.134721,-0.145918,-0.216656,-0.035429,-0.174292,-0.262093,-0.014664,0.659276,...,-0.049628,-0.039607,-0.023518,-0.024163,-2.932903,-0.045408,-0.022172,-0.135021,0.56319,-0.56319


## 7. Dividir el Dataset en 80/20 (Entrenamiento y Prueba)
Finalmente apartaremos el 80% de nuestros datos para que el modelo aprenda y un 20% para hacerle un examen y evaluarlo.
En lugar de una caja negra, apartamos las filas simplemente cortando las tablas basándonos en el porcentaje.


In [8]:
# Extraemos la última columna para usarla como objetivo de predicción (Variable Y o Target)
objetivo_col = datos_normalizados.columns[-1]
caracteristicas = datos_normalizados.drop(objetivo_col, axis=1)
objetivo = datos_normalizados[objetivo_col]

# --------- DIVISIÓN A MANO (80% Entrenamiento, 20% Prueba) ---------

# 1. Calculamos cuánto es el 80% entero de nuestras filas
total_filas = len(datos_normalizados)
ochenta_por_ciento = int(total_filas * 0.8)

# 2. Cortamos las tablas usando posiciones desde 0 hasta el 80%
X_entrenamiento = caracteristicas.iloc[:ochenta_por_ciento]
y_entrenamiento = objetivo.iloc[:ochenta_por_ciento]

# 3. Y para el resto (el 20% restante), usamos desde el 80% en adelante
X_prueba = caracteristicas.iloc[ochenta_por_ciento:]
y_prueba = objetivo.iloc[ochenta_por_ciento:]

print('Total datos Original:', total_filas, 'filas')
print('Datos para Entrenar (80%):', len(X_entrenamiento), 'filas')
print('Datos para Probar (20%):', len(X_prueba), 'filas')
print('\n¡El Dataset está preparado puramente usando lógica matemática!')


Total datos Original: 32561 filas
Datos para Entrenar (80%): 26048 filas
Datos para Probar (20%): 6513 filas

¡El Dataset está preparado puramente usando lógica matemática!
